[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sktime/pytorch-forecasting/blob/main/docs/source/tutorials/ptf_V2_example.ipynb)

*(This tutorial can be run interactively in Google Colab)*

In [1]:
# Uncomment the line below to install the library in Google Colab
# !pip install pytorch-forecasting

# Example Notebook for a basic vignette for `pytorch-forecasting v2` Model Training and Inference

<div class="alert alert-block alert-info">
:warning: The "Data Pipeline" showcased here is part of an experimental rework of the `pytorch-forecasting` data layer, planned for release in v2.0.0. The API is currently unstable and subject to change without prior notice. This notebook serves as a basic demonstration of the intended workflow and is not recommended for use in production environments. Feedback and suggestions are highly encouraged — please share them in <a href="https://github.com/sktime/pytorch-forecasting/issues/1736">issue 1736</a>.
</div>


In this notebook, we demonstrate how to train and evaluate the **Temporal Fusion Transformer (TFT)** using the new `TimeSeries` and `DataModule` API from the v2 pipeline.
We can do this in 2 ways:
1. **High-level forecaster API:**

    This approach handles data loading, dataloader creation, and model training internally. It provides a simple, `scikit-learn`-like `fit` → `predict` workflow.
    You pass the model's own parameters directly to the forecaster, together with a configured `datamodule` and a `lightning.Trainer`, so callbacks and training options stay fully under your control.

2. **Low-level 3-stage pipeline**:
This involves explicitly constructing:
    * a `TimeSeries` object

    * a `DataModule`

    * the model (e.g., `TFT`)
    
    This workflow is ideal if you need custom setups such as custom training loops, callbacks, or advanced data preprocessing.
    It requires a deeper understanding of how the three layers (TimeSeries, DataModule, and the model) interact, but offers maximum flexibility.

## Create Synthetic data
We generate a synthetic dataset using `load_toydata` that creates a `pandas` DataFrame with just numerical values as for now **the pipeline assumes the data to be numerical only**.

In [1]:
from pytorch_forecasting.data.examples import load_toydata

In [2]:
num_series = 100  # Number of individual time series to generate
seq_length = 50  # Length of each time series
data_df = load_toydata(num_series, seq_length)
data_df.head()

,series_id,time_idx,x,y,category,future_known_feature,static_feature,static_feature_cat
0,0,0,0.020719,0.146227,0,1.000000,0.974826,0
1,0,1,0.146227,0.457556,0,0.995004,0.974826,0
2,0,2,0.457556,0.644491,0,0.980067,0.974826,0
3,0,3,0.644491,0.737403,0,0.955336,0.974826,0
4,0,4,0.737403,0.830147,0,0.921061,0.974826,0


## High-level API: `Forecaster`

### Steps
* Create the `TimeSeries` object
* Create the `datamodule` and the `trainer`
* Create the `forecaster` object
* perform `forecaster.fit` and `forecaster.predict`.

###  Create Dataset object

`TimeSeries` returns the raw data in terms of tensors .

---

`TimeSeries` dataset's Key arguments:
- `data`: DataFrame with sequence data.
- `time`: integer typed column denoting the time index within `data`.
- `target`:  Column(s) in `data` denoting the forecasting target.
- `group`: List of column names identifying a time series instance within `data`.
- `num`: List of numerical features.
- `cat`: List of categorical features.
- `known`: Features known in future
- `unknown`: Features not known in the future
- `static`: List of variables that do not change over time,

In [3]:
from pytorch_forecasting.data.timeseries import TimeSeries

In [4]:
# create `TimeSeries` dataset that returns the raw data in terms of tensors
dataset = TimeSeries(
    data=data_df,
    time="time_idx",
    target="y",
    group=["series_id"],
    num=["x", "future_known_feature", "static_feature"],
    cat=["category", "static_feature_cat"],
    known=["future_known_feature"],
    unknown=["x", "category"],
    static=["static_feature", "static_feature_cat"],
)

/mnt/c/Users/hirwa/Desktop/open/pytorch-forecasting/pytorch_forecasting/data/timeseries/_timeseries_v2.py:129: UserWarning: TimeSeries is part of an experimental rework of the pytorch-forecasting data layer, scheduled for release with v2.0.0. The API is not stable and may change without prior warning. For beta testing, but not for stable production use. Feedback and suggestions are very welcome in pytorch-forecasting issue 1736, https://github.com/sktime/pytorch-forecasting/issues/1736
  warn(


### Create the `datamodule` and the `trainer`

In [5]:
from lightning.pytorch import Trainer
from sklearn.preprocessing import StandardScaler

from pytorch_forecasting.data.data_module import EncoderDecoderTimeSeriesDataModule
from pytorch_forecasting.data.encoders import (
    NaNLabelEncoder,
    TorchNormalizer,
)
from pytorch_forecasting.metrics import MAE, SMAPE

Here we use `EncoderDecoderTimeSeriesDataModule`.

The datamodule is created **without data** — `forecaster.fit` attaches the `TimeSeries` to it, and `forecaster.predict` reuses the transforms fitted during `fit`.

`EncoderDecoderTimeSeriesDataModule` key arguments:
- `max_encoder_length` : Maximum length of the encoder input sequence.
- `max_prediction_length` : Maximum length of the decoder output sequence.
- `batch_size` : Batch size for DataLoader.
- `categorical_encoders` :  Dictionary of categorical encoders.
- `scalers` : Dictionary of feature scalers.
- `target_normalizer`: Normalizer for the target variable.

In [6]:
datamodule = EncoderDecoderTimeSeriesDataModule(
    max_encoder_length=30,
    max_prediction_length=1,
    batch_size=32,
    categorical_encoders={
        "category": NaNLabelEncoder(add_nan=True),
        "static_feature_cat": NaNLabelEncoder(add_nan=True),
    },
    scalers={
        "x": StandardScaler(),
        "future_known_feature": StandardScaler(),
        "static_feature": StandardScaler(),
    },
    target_normalizer=TorchNormalizer(),
)

/tmp/ipykernel_97547/3149672647.py:1: UserWarning: EncoderDecoderTimeSeriesDataModule is part of an experimental rework of the pytorch-forecasting data layer, scheduled for release with v2.0.0. The API is not stable and may change without prior warning. For beta testing, but not for stable production use. Feedback and suggestions are very welcome in pytorch-forecasting issue 1736, https://github.com/sktime/pytorch-forecasting/issues/1736
  datamodule = EncoderDecoderTimeSeriesDataModule(


We use a `lightning.Trainer` to control the training loop:

In [7]:
trainer = Trainer(
    max_epochs=5,
    accelerator="auto",
    devices=1,
    enable_progress_bar=True,
    log_every_n_steps=10,
)

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores


We use the `TFT` model in this tutorial, through its forecaster class `TFTForecaster`.
Its parameters mirror those of the underlying `TFT` model, so there is no separate `model_cfg` to build.

In [8]:
from pytorch_forecasting.models.temporal_fusion_transformer import TFTForecaster

### Create the `forecaster` object

The forecaster acts as a wrapper around the whole ML pipeline in `pytorch-forecasting`: we define the forecaster once and then use `forecaster.fit` and `forecaster.predict` to perform the "fit" and "predict" mechanisms.

In [9]:
forecaster = TFTForecaster(
    hidden_size=64,
    num_layers=2,
    attention_head_size=4,
    dropout=0.1,
    loss=MAE(),
    logging_metrics=[MAE(), SMAPE()],
    optimizer="adam",
    optimizer_params={"lr": 1e-3},
    lr_scheduler="reduce_lr_on_plateau",
    lr_scheduler_params={"mode": "min", "factor": 0.1, "patience": 10},
    datamodule=datamodule,
    trainer=trainer,
)

In [10]:
# `fit` saves the best checkpoint and returns its path
ckpt_path = forecaster.fit(dataset)

/mnt/c/Users/hirwa/Desktop/open/pytorch-forecasting/pytorch_forecasting/data/data_module/base_data_module.py:220: UserWarning: EncoderDecoderTimeSeriesDataModule is part of an experimental rework of the pytorch-forecasting data layer, scheduled for release with v2.0.0. The API is not stable and may change without prior warning. For beta testing, but not for stable production use. Feedback and suggestions are very welcome in pytorch-forecasting issue 1736, https://github.com/sktime/pytorch-forecasting/issues/1736
  return type(self)(time_series=data, **self._init_kwargs)
/mnt/c/Users/hirwa/Desktop/open/pytorch-forecasting/pytorch_forecasting/models/base/_base_model_v2.py:93: UserWarning: The Model 'TFT' is part of an experimental reworkof the pytorch-forecasting model layer, scheduled for release with v2.0.0. The API is not stable and may change without prior warning. This class is intended for beta testing and as a basic skeleton, but not for stable production use. Feedback and suggest

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/mnt/c/Users/hirwa/Desktop/open/pytorch-forecasting/env/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.
/mnt/c/Users/hirwa/Desktop/open/pytorch-forecasting/env/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=5` reached.


Artifacts saved in: /mnt/c/Users/hirwa/Desktop/open/pytorch-forecasting/docs/source/tutorials/checkpoints


#### Output

`forecaster.predict` returns a `TimeSeries` of predictions, with one row per forecast step.
Its `data` frame holds:

- `_series`: index of the source series
- `_time_idx`: position of the forecast step within that series
- `y`: the predicted value, named after the target
- `y_actual`: the observed value, present because `"y"` was passed in `return_info`

Values are in the space of the fitted target normalizer, so `y` and `y_actual` are directly comparable.

In [11]:
preds = forecaster.predict(dataset, return_info=["index", "x", "y"])

/mnt/c/Users/hirwa/Desktop/open/pytorch-forecasting/pytorch_forecasting/data/data_module/base_data_module.py:220: UserWarning: EncoderDecoderTimeSeriesDataModule is part of an experimental rework of the pytorch-forecasting data layer, scheduled for release with v2.0.0. The API is not stable and may change without prior warning. For beta testing, but not for stable production use. Feedback and suggestions are very welcome in pytorch-forecasting issue 1736, https://github.com/sktime/pytorch-forecasting/issues/1736
  return type(self)(time_series=data, **self._init_kwargs)
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/mnt/c/Users/hirwa/Desktop/open/pytorch-forecasting/env/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


Predicting: |          | 0/? [00:00<?, ?it/s]

/mnt/c/Users/hirwa/Desktop/open/pytorch-forecasting/pytorch_forecasting/data/timeseries/_timeseries_v2.py:129: UserWarning: TimeSeries is part of an experimental rework of the pytorch-forecasting data layer, scheduled for release with v2.0.0. The API is not stable and may change without prior warning. For beta testing, but not for stable production use. Feedback and suggestions are very welcome in pytorch-forecasting issue 1736, https://github.com/sktime/pytorch-forecasting/issues/1736
  warn(


In [12]:
preds.data.head()

,_series,_time_idx,y,y_actual
0,0,30,-0.219742,-0.156817
1,0,31,-0.011999,-0.001194
2,0,32,0.294100,0.088528
3,0,33,0.572353,0.468169
4,0,34,0.778117,0.584692


In [13]:
print("First Predicted Value:")
print("Prediction:", preds.data["y"].iloc[0])
print("Actual:", preds.data["y_actual"].iloc[0])

First Predicted Value:
Prediction: -0.21974245
Actual: -0.15681686


## 3-stage pipeline




### Steps

1. Create `TimeSeries` Dataset object
2. Create DataModule object
3. Initialize, Train & Run Inference with the Model




####  Create Dataset & DataModule

- `TimeSeries` returns the raw data in terms of tensors .
- `DataModule` wraps the dataset, handles splits, preprocessing, batching, and exposes `metadata` for the model initialisation.



#### Initialize the Model

We initialize the TFT model using the `metadata` provided by the `DataModule`. This metadata includes all required dimensional info for the encoder, decoder, and static inputs.



#### Train the Model

We use a `Trainer` from PyTorch Lightning to train the model

#### Run Inference

After training, we can make predictions using the trained model


### 1. Create the dataset
We create a `TimeSeries` dataset instance that returns the raw data in terms of tensors, then this "raw data" is sent to the `data_module`that will internally handle the dataloaders and preprocessing

`TimeSeries` dataset's Key arguments:
- `data`: DataFrame with sequence data.
- `time`: integer typed column denoting the time index within `data`.
- `target`:  Column(s) in `data` denoting the forecasting target.
- `group`: List of column names identifying a time series instance within `data`.
- `num`: List of numerical features.
- `cat`: List of categorical features.
- `known`: Features known in future
- `unknown`: Features not known in the future
- `static`: List of variables that do not change over time,

In [14]:
from pytorch_forecasting.data.timeseries import TimeSeries

In [15]:
# create `TimeSeries` dataset that returns the raw data in terms of tensors
dataset = TimeSeries(
    data=data_df,
    time="time_idx",
    target="y",
    group=["series_id"],
    num=["x", "future_known_feature", "static_feature"],
    cat=["category", "static_feature_cat"],
    known=["future_known_feature"],
    unknown=["x", "category"],
    static=["static_feature", "static_feature_cat"],
)

/mnt/c/Users/hirwa/Desktop/open/pytorch-forecasting/pytorch_forecasting/data/timeseries/_timeseries_v2.py:129: UserWarning: TimeSeries is part of an experimental rework of the pytorch-forecasting data layer, scheduled for release with v2.0.0. The API is not stable and may change without prior warning. For beta testing, but not for stable production use. Feedback and suggestions are very welcome in pytorch-forecasting issue 1736, https://github.com/sktime/pytorch-forecasting/issues/1736
  warn(


### 2. Create datamodule

`EncoderDecoderTimeSeriesDataModule` key arguments:
- `time_series`: `TimeSeries` dataset instance
- `max_encoder_length` : Maximum length of the encoder input sequence.
- `max_prediction_length` : Maximum length of the decoder output sequence.
- `batch_size` : Batch size for DataLoader.
- `categorical_encoders` :  Dictionary of categorical encoders.
- `scalers` : Dictionary of feature scalers.
- `target_normalizer`: Normalizer for the target variable.

In [16]:
from sklearn.preprocessing import StandardScaler
from pytorch_forecasting.data.data_module import EncoderDecoderTimeSeriesDataModule
from pytorch_forecasting.data.encoders import (
    NaNLabelEncoder,
    TorchNormalizer,
)

In [17]:
# create the `data_module` that handles the dataloaders and preprocessing
data_module = EncoderDecoderTimeSeriesDataModule(
    time_series=dataset,
    max_encoder_length=30,
    max_prediction_length=1,
    batch_size=32,
    categorical_encoders={
        "category": NaNLabelEncoder(add_nan=True),
        "static_feature_cat": NaNLabelEncoder(add_nan=True),
    },
    scalers={
        "x": StandardScaler(),
        "future_known_feature": StandardScaler(),
        "static_feature": StandardScaler(),
    },
    target_normalizer=TorchNormalizer(),
)

/tmp/ipykernel_97547/1675030273.py:2: UserWarning: EncoderDecoderTimeSeriesDataModule is part of an experimental rework of the pytorch-forecasting data layer, scheduled for release with v2.0.0. The API is not stable and may change without prior warning. For beta testing, but not for stable production use. Feedback and suggestions are very welcome in pytorch-forecasting issue 1736, https://github.com/sktime/pytorch-forecasting/issues/1736
  data_module = EncoderDecoderTimeSeriesDataModule(


### 3. Initialise and train the model

To initialise the model you now don't have to pass arguments like `encoder_cont`, `decoder_cont` etc as they are calculated internally using the [`metadata`](https://github.com/sktime/pytorch-forecasting/blob/main/.../site-packages/pytorch_forecasting/data/data_module/base_data_module.py) property of `EncoderDecoderTimeSeriesDataModule`. But you still have to pass other params like `loss`, `optimizer` etc


```python
model = TFT(
    loss=nn.MSELoss(),
    logging_metrics=[MAE(), SMAPE()],
    metadata=data_module.metadata,  # <-- crucial for model setup
    ...
)
```

The `metadata` includes:
- `max_encoder_length`, `max_prediction_length`
- number of continuous/categorical variables in encoder/decoder
- number of static features

These are used to configure internal layers like `encoder_cont`, `decoder_cat`, etc.


In [18]:
from pytorch_forecasting.metrics import MAE, SMAPE
from pytorch_forecasting.models.temporal_fusion_transformer._tft_v2 import TFT

In [19]:
# Initialise the Model
model = TFT(
    loss=MAE(),
    logging_metrics=[MAE(), SMAPE()],
    optimizer="adam",
    optimizer_params={"lr": 1e-3},
    lr_scheduler="reduce_lr_on_plateau",
    lr_scheduler_params={"mode": "min", "factor": 0.1, "patience": 10},
    hidden_size=64,
    num_layers=2,
    attention_head_size=4,
    dropout=0.1,
    metadata=data_module.metadata,  # pass the metadata from the datamodule to the model
    # to initialise important params like `encoder_cont` etc
)

/mnt/c/Users/hirwa/Desktop/open/pytorch-forecasting/pytorch_forecasting/models/base/_base_model_v2.py:93: UserWarning: The Model 'TFT' is part of an experimental reworkof the pytorch-forecasting model layer, scheduled for release with v2.0.0. The API is not stable and may change without prior warning. This class is intended for beta testing and as a basic skeleton, but not for stable production use. Feedback and suggestions are very welcome in pytorch-forecasting issue 1736, https://github.com/sktime/pytorch-forecasting/issues/1736
  warn(


We use a `Trainer` from PyTorch Lightning to train the model:

```python
trainer = Trainer(max_epochs=5, ...)
trainer.fit(model, data_module)
```

The `Trainer`:
- Pulls data from `data_module`
- Handles device placement
- Logs training progress and metrics


In [20]:
from lightning.pytorch import Trainer

In [21]:
# Train the model
print("\nTraining model...")
trainer = Trainer(
    max_epochs=5,
    accelerator="auto",
    devices=1,
    enable_progress_bar=True,
    log_every_n_steps=10,
)

trainer.fit(model, data_module)

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.



Training model...


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name                  | Type               | Params | Mode  | FLOPs
-----------------------------------------------------------------------------
0 | loss                  | MAE                | 0      | train | 0    
1 | logging_metrics       | ModuleList         | 0      | train | 0    
2 | encoder_var_selection | Sequential         | 709    | train | 0    
3 | decoder_var_selection | Sequential         | 193    | train | 0    
4 | static_context_linear | Linear             | 192    | train | 0    
5 | lstm_encoder          | LSTM               | 51.5 K | train | 0    
6 | lstm_decoder          | LSTM               | 50.4 K | train | 0    
7 | self_attention        | MultiheadAttention | 16.6 K | train | 0    
8 | pre_output            | Linear             | 4.2 K  | train | 0    
9 | output_layer          | Linear             | 65     | train | 0    
-------

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/mnt/c/Users/hirwa/Desktop/open/pytorch-forecasting/env/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.
/mnt/c/Users/hirwa/Desktop/open/pytorch-forecasting/env/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


Training: |          | 0/? [00:00<?, ?it/s]

... (2 outputs left out)

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=5` reached.



#### Output
Output of TFT model is a `dict` with key `prediction`:

- `y_pred["prediction"]`: Tensor of shape `(batch_size, prediction_length, output_size)`


In [22]:
data_module.setup(stage="test")
test_dataloader = data_module.test_dataloader()

In [23]:
preds = model.predict(test_dataloader, return_info=["index", "x", "y"])

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/mnt/c/Users/hirwa/Desktop/open/pytorch-forecasting/env/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


Predicting: |          | 0/? [00:00<?, ?it/s]

In [24]:
print("First Predicted Value:")
print("Prediction:", preds["prediction"][0].item())
print("Actual:", preds["y"][0].item())

First Predicted Value:
Prediction: -0.3185421824455261
Actual: -0.42513012886047363
